# Scene completion
The pipeline to complete the partial environment scan.

In [ ]:
import numpy as np
import trimesh
import os
import sys
sys.path.insert(0, '../')
import drm
from PIL import Image
import torch
from diffusers import StableDiffusionInpaintPipeline
import numpy as np
from pathlib import Path

## Load pointcloud

In [ ]:
pcdPath = "/home/jvermandere/projects/DRM/_output/segmented_points/isolated_points.txt"
pointsColors = np.loadtxt(pcdPath, dtype=np.float32).reshape(-1, 6)
cloud = trimesh.points.PointCloud(pointsColors[:, :3], colors=pointsColors[:, 3:6]/255)
scene = trimesh.Scene(cloud)
scene.show()

## Ransac itterative plane detection

In [ ]:
min_points = 100  # minimum points to consider a plane
remaining_points = cloud.vertices.copy()
remaining_colors = cloud.colors.copy() if cloud.colors is not None else None

planes = []  # list of trimesh point clouds for each plane

while len(remaining_points) >= min_points:
    plane_model, inliers = drm.ransac_plane_trimesh(remaining_points,
                                        num_iterations=1000,
                                        distance_threshold=0.01)

    if len(inliers) < min_points:
        print("No more large planes detected.")
        break

    # Extract plane points and colors
    plane_pts = remaining_points[inliers]
    plane_colors = remaining_colors[inliers] if remaining_colors is not None else None
    plane_pc = trimesh.points.PointCloud(vertices=plane_pts, colors=plane_colors)
    planes.append(plane_pc)

    # Remove plane points from remaining points
    mask = np.ones(len(remaining_points), dtype=bool)
    mask[inliers] = False
    remaining_points = remaining_points[mask]
    if remaining_colors is not None:
        remaining_colors = remaining_colors[mask]

# Remaining points as a separate point cloud
if len(remaining_points) > 0:
    remaining_pc = trimesh.points.PointCloud(vertices=remaining_points,
                                             colors=remaining_colors)
else:
    remaining_pc = None

print(f"Extracted {len(planes)} planes.")

In [ ]:
scene = drm.visualize_pointclouds_random_colors(planes)
scene.show()

## Inpainting all the planes

In [ ]:
# Load inpainting model
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting"
).to("cuda")


In [ ]:
# Paint in all the planes one by one
filled_planes = []
for i in range(len(planes)):
    #fill in the planes
    filled_plane = drm.fill_plane_holes(planes[i], target_density=0.01)
    # get the 2D images of the partial plane and to-be-filled-in area of the new plane
    img, mask, filled_coords = drm.project_planes_with_infill_mask(planes[i], filled_plane, resolution=512, point_radius=3)
    # paint in the image
    image_pil = Image.fromarray(img.astype(np.uint8)).convert("RGB").resize((512, 512))
    mask_pil = Image.fromarray((mask * 255).astype(np.uint8)).convert("L").resize((512, 512))
    # Run inpainting
    inpaintedImage = pipe(prompt="paint",image=image_pil,mask_image=mask_pil).images[0]
    #
    texture = np.asarray(inpaintedImage)
    colors = texture[filled_coords[:,1], filled_coords[:,0]]
    filled_plane.colors = np.hstack([colors,np.full((len(colors),1),255)])
    filled_planes.append(filled_plane)


In [ ]:
scene = trimesh.Scene(filled_planes[:4])
scene.show()

### Save the completed pointcloud

In [ ]:
all_vertices = []
all_colors = []

for pc in filled_planes[:4]:
    all_vertices.append(pc.vertices)

    if pc.colors is not None:
        all_colors.append(pc.colors)
    else:
        # default white
        all_colors.append(np.full((len(pc.vertices), 4), 255, dtype=np.uint8))

vertices = np.vstack(all_vertices)
colors = np.vstack(all_colors)

merged_pc = trimesh.points.PointCloud(vertices=vertices, colors=colors)

merged_pc.export(Path(pcdPath).with_suffix(".reconstructed_2.ply"))

## Inpainting one plane

In [ ]:
filled_planes = []
for plane in planes:
    filled_plane = drm.fill_plane_holes(plane, target_density=0.01)
    filled_planes.append(filled_plane)

scene = trimesh.Scene(filled_planes[:4])
scene.show()

In [ ]:
import matplotlib.pyplot as plt

# filled_plane = fill_plane_holes_with_colors(plane)
img, mask, filled_coords = drm.project_planes_with_infill_mask(planes[3], filled_planes[3], resolution=512, point_radius=0)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Original Plane Colors")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(mask, cmap='gray')
plt.title("Filled Areas / Holes")
plt.axis("off")

plt.show()

In [ ]:
from PIL import Image
import torch
from diffusers import StableDiffusionInpaintPipeline
import numpy as np
# Load inpainting model
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting"
).to("cuda")



In [ ]:

# Convert numpy arrays
image_pil = Image.fromarray(img.astype(np.uint8)).convert("RGB")
mask_pil = Image.fromarray((mask * 255).astype(np.uint8)).convert("L")

# Resize both to same dimensions (512x512 is safe for SD)
image_pil = image_pil.resize((512, 512))
mask_pil = mask_pil.resize((512, 512))



#prompt = "paint in the masked area to fill in the missing regions looking at the rest of the colors in the image to match the texture, ignore eveything that is black"
prompt = "paint"

# Run inpainting
result = pipe(
    prompt=prompt,
    image=image_pil,
    mask_image=mask_pil
).images[0]


In [ ]:
result

In [ ]:
def apply_texture_to_plane(filled_pc, inpainted_img, filled_coords):

    img = np.asarray(inpainted_img)

    new_colors = np.zeros((len(filled_coords), 4), dtype=np.uint8)

    for i, (x,y) in enumerate(filled_coords):

        color = img[y, x]

        if len(color) == 3:
            new_colors[i] = np.append(color, 255)
        else:
            new_colors[i] = color

    filled_pc.colors = new_colors

    return filled_pc

In [ ]:
filled_pc = apply_texture_to_plane(
    filled_planes[3],
    result,
    filled_coords
)

In [ ]:
scene = trimesh.Scene(filled_pc)
scene.show()